# Assignment 1: Information Retrieval

## Overview
In this assignment you will implement a BM25 retrieval system from scratch.
Given a text query, your system must find the most relevant documents from the **Cranfield Collection** — 
a foundational corpus for evaluating information retrieval systems containing:
- **1,400** documents
- **225** queries
- **1,837** relevance judgments (qrels)

## Rules
- You may only use **built-in Python libraries** plus `numpy` and `pandas`
- Full-packaged retrieval solutions (e.g. `rank_bm25`) are **not permitted**
- Do **not** modify function signatures or the skeleton class
- Do **not** change cells marked **do not edit**

## Deadline
**March 19, 23:59 CET**

## How to Use This Notebook
1. Run imports and BM25 Skeleton class first — always
2. Implement each task in the cells marked **TODO: your code here** and answer questions marked **TODO**
3. If you restart the kernel, re-run all cells from top to bottom

## Load the data

In [1]:
import ir_datasets
dataset = ir_datasets.load("cranfield")

### Inspect the queries

In [2]:
for query in dataset.queries_iter():
    print(query) # namedtuple<query_id, text>
    break

[INFO] [starting] http://ir.dcs.gla.ac.uk/resources/test_collections/cran/cran.tar.gz
[INFO] [finished] http://ir.dcs.gla.ac.uk/resources/test_collections/cran/cran.tar.gz: [00:00] [507kB] [3.00MB/s]
                                                                                               

GenericQuery(query_id='1', text='what similarity laws must be obeyed when constructing aeroelastic models\nof heated high speed aircraft .')


### Inspect the documents

In [3]:
for doc in dataset.docs_iter():
    print(doc) # namedtuple<doc_id, title, text, author, bib>
    break

CranfieldDoc(doc_id='1', title='experimental investigation of the aerodynamics of a\nwing in a slipstream .', text='experimental investigation of the aerodynamics of a\nwing in a slipstream .\n  an experimental study of a wing in a propeller slipstream was\nmade in order to determine the spanwise distribution of the lift\nincrease due to slipstream at different angles of attack of the wing\nand at different free stream to slipstream velocity ratios .  the\nresults were intended in part as an evaluation basis for different\ntheoretical treatments of this problem .\n  the comparative span loading curves, together with\nsupporting evidence, showed that a substantial part of the lift increment\nproduced by the slipstream was due to a /destalling/ or\nboundary-layer-control effect .  the integrated remaining lift\nincrement, after subtracting this destalling lift, was found to agree\nwell with a potential flow theory .\n  an empirical evaluation of the destalling effects was made for\nthe s

### Inspect the query-document relevance judgments

In [4]:
for qrel in dataset.qrels_iter():
    print(qrel) # namedtuple<query_id, doc_id, relevance, iteration>
    break

TrecQrel(query_id='1', doc_id='184', relevance=2, iteration='0')


## Setup

The cells below set up the imports and define the BM25 skeleton class. 
You do not need to modify anything here — simply run both cells before starting Task 1.

In [5]:
from __future__ import annotations

import math
import re
from collections import Counter, defaultdict
from dataclasses import dataclass
from typing import Dict, Iterable, List, Tuple, Set, Optional

import numpy as np

# NLTK for tokenization
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
nltk.download("stopwords", quiet=True)

True

**Note:** The `BM25` class below contains all the methods you will implement throughout this assignment.
Each task will ask you to implement one or more of these methods and attach them to the class.

In [ ]:
# BM25 Skeleton (run this first, do not edit)

class BM25:
    """
    BM25 class:
      - tokenizer: tokenize(text) -> List[str]
      - retriever: retrieve(query_text, k) -> ranked list
      - evaluator: evaluate(metric, k) and grid_search over k1, b

    Brute-force scoring over all documents (no inverted index).
    """
    def __init__(self, k1: float = 1.2, b: float = 0.75, remove_stopwords: bool = True):
        
        # BM25 Hyperparameters
        self.k1: float = k1                    # term frequency saturation
        self.b: float = b                      # document length normalization
        self.remove_stopwords: bool = remove_stopwords
        
        # Initializations for tokenization
        self._stemmer = PorterStemmer()      # work stemming
        self._stopwords = set(stopwords.words("english")) if self.remove_stopwords else set() # stopwords set
        
        # Corpus Statistics (populated by build())
        self.N: int = 0                        # total number of documents
        self.avgdl: float = 0.0               # average document length
        self.df: Dict[str, int] = {}          # term -> number of docs containing term
        self.doc_len: Dict[str, int] = {}     # doc_id -> document length
        self.doc_tf: Dict[str, Counter] = {}  # doc_id -> term frequency Counter
        self.docs_store: Dict[str, Dict[str, str]] = {}  # doc_id -> {"title": ..., "text": ...}

    def tokenize(self, text: str) -> List[str]: raise NotImplementedError
    def build(self, docs: Iterable) -> None: raise NotImplementedError
    def idf(self, term: str) -> float: raise NotImplementedError
    def score(self, query_tokens: List[str], doc_id: str) -> float: raise NotImplementedError
    def retrieve(self, query_text: str, k: int = 10) -> List[Tuple[str, float]]: raise NotImplementedError
    def evaluate(self, queries_iter: Iterable, qrels_iter: Iterable, k: int = 10, metrics: Optional[List[str]] = None) -> Dict[str, float]: raise NotImplementedError
    def grid_search(self, queries_iter_factory, qrels_iter_factory, k: int = 10) -> List[Dict]: raise NotImplementedError



## Task 1: Tokenizer 

Implement a tokenizer that converts a text into a list of keywords: tokenize(text) -> List[str].

In [ ]:
def tokenize(self, text: str) -> List[str]:
    """
    Convert raw text into a list of processed tokens.

    Requirements:
      1. Lowercase everything         e.g.  "UZH"                       -> ["uzh"]
      2. Extract alphanumeric words   e.g.  "hello, world 2026!"        -> ["hello", "world", "2026"]
      3. Remove stopwords             e.g.  "the document has a title"  -> ["document", "title"]
      4. Apply Porter stemming        e.g.  "working worked works"      -> ["work", "work", "work"]
      5. Drop empty tokens            e.g.  " retrieval "               -> ["retrieval"]

    Args:
        text: raw input string
    Returns:
        List of processed token strings

    Hints:
        - Perform text normalization using Python string methods.
        - Use the `re` module to extract alphanumeric tokens.
        - For stemming and stopword removal, use the initialized stemmer and stopword list from the BM25 class constructor.

    """
    # Step 1: Lowercase
    text = text.lower()
    
    # Step 2: Extract alphanumeric words
    tokens = re.findall(r'\b\w+\b', text)
    
    # Step 3 & 4: Remove stopwords and apply stemming
    processed_tokens = []
    for token in tokens:
        if token not in self._stopwords:
            stemmed_token = self._stemmer.stem(token)
            if stemmed_token:  # Ensure we don't add empty tokens
                processed_tokens.append(stemmed_token)
    
    return processed_tokens

    
# attach to BM25 class
BM25.tokenize = tokenize

In [8]:
# ===============================================
# NOTE: Personal Testing Only — NOT Part of Assignment
# ===============================================
#
# This cell contains some basic example test cases for the `tokenize()` method.
# It is meant to help you check your own implementation.
# 
# It does NOT count toward your grade and is safe to run multiple times.
# You can modify it to test additional examples if you want.


bm25_test = BM25(remove_stopwords=True)

# Test 1: Lowercasing + stemming
assert bm25_test.tokenize("Working worked works") == ["work", "work", "work"]

# Test 2: Alphanumeric extraction + punctuation removal
assert bm25_test.tokenize("Hello, world 2026!") == ["hello", "world", "2026"]

# Test 3: Stopword removal + stemming
assert bm25_test.tokenize("The document has a title") == ["document", "titl"]

# Test 4: Mixed case + symbols + stemming
assert bm25_test.tokenize("Machine-Learning @ UZH :)") == ["machin", "learn", "uzh"]

print("All tokenizer tests passed")

All tokenizer tests passed


## Task 2: BM25 Retriever

Implement the document retriever using BM25 that finds the best top-k documents for a given query.

### Task 2a: Build Index

Implement `build()` to precompute the corpus statistics needed for BM25 scoring.

Once complete, the following attributes will be populated and used by `score()` and `retrieve()` in the next steps.

In [ ]:

def build(self, docs_iter: Iterable) -> None:
    """
    Iterate over all documents and precompute corpus statistics needed for BM25.

    Args:
        docs_iter: iterable of Cranfield documents, each a namedtuple with fields:
                   (doc_id, title, text, author, bib)
    
    Requirements: 
        After calling build(), the following must be populated:
            self.N        : int                      — total number of documents
            self.avgdl    : float                    — average document length
            self.df       : Dict[str, int]           — term -> number of docs containing term
            self.doc_len  : Dict[str, int]           — doc_id -> document length
            self.doc_tf   : Dict[str, Counter]       — doc_id -> term frequencies per document
            self.docs_store: Dict[str, Dict[str, str]] — doc_id:str -> {"title": ..., "text": ...} (raw title & text per document)

    Note:
        Per-document steps (repeat for each document):
            1. Extract title and text safely
            2. Concatenate title and text, then tokenize using self.tokenize()
            3. Compute term frequencies  → store in self.doc_tf
            4. Compute document length → store in self.doc_len
            5. Store raw title and text → store in self.docs_store

        Corpus-level steps (after all documents):
            6. Count how many documents each term appears in → store in self.df
            (each term is counted once per document, not once per occurrence)
            7. Compute self.avgdl and self.N

    """
    # Step 1-5: Process each document
    for doc in docs_iter:
        doc_id = doc.doc_id
        title = doc.title if doc.title else ""
        text = doc.text if doc.text else ""
        
        # Concatenate title and text
        full_text = f"{title} {text}"
        
        # Tokenize
        tokens = self.tokenize(full_text)
        
        # Compute term frequencies
        tf_counter = Counter(tokens)
        self.doc_tf[doc_id] = tf_counter
        
        # Compute document length
        doc_length = sum(tf_counter.values())
        self.doc_len[doc_id] = doc_length
        
        # Store raw title and text
        self.docs_store[doc_id] = {"title": title, "text": text}
    
    # Step 6: Compute document frequency for each term
    df_counter = Counter()
    for tf in self.doc_tf.values():
        unique_terms = set(tf.keys())
        df_counter.update(unique_terms)
    self.df = dict(df_counter)
    
    # Step 7: Compute average document length and total number of documents
    self.N = len(self.doc_tf)
    self.avgdl = np.mean(list(self.doc_len.values())) if self.N > 0 else 0.0

# attach to BM25 class
BM25.build = build

### Task 2b: BM25 Scoring

Now that the corpus statistics can be precomputed, implement the two core scoring functions of BM25.

- `idf(term)` computes how rare a term is across the corpus
- `score(query_tokens, doc_id)` combines IDF with term frequency to compute a BM25 score for a query-document pair


In [10]:
def idf(self, term: str) -> float:
    """
    Compute Inverse Document Frequency with standard smoothing.

    Args:
        term: a single token str

    Returns:
        float — IDF score for the term

    Formula:
        log( (N - df + 0.5) / (df + 0.5) + 1 )

    """
    N = self.N
    df = self.df.get(term, 0)
    idf_score = math.log((N - df + 0.5) / (df + 0.5) + 1)
    return idf_score
# attach to BM25 class
BM25.idf = idf


def score(self, query_tokens: List[str], doc_id: str) -> float:
    """
    Compute BM25 score for a query against a single document.

    Args:
        query_tokens: list of tokenized query terms
        doc_id: document to score against

    Returns:
        float — BM25 score for this query-document pair

    Formula (sum over query terms):
        score(q, d) = sum over t in q [ IDF(t) * (f * (k1 + 1)) / (f + k1 * (1 - b + b * dl/avg_dl)) ]

    where:
        q    = query
        d    = document
        t    = a query term
        f    = term frequency of t in document
        dl   = length of the document
        k1   = self.k1
        b    = self.b

    Hints:
        - skip terms with frequency 0
        - return 0.0 for empty documents
        - sum over query terms (you can keep duplicates, but set() is common and stable for students)
    """
    score = 0.0
    doc_tf = self.doc_tf.get(doc_id, Counter())
    doc_length = self.doc_len.get(doc_id, 0)
    
    for term in query_tokens:
        f = doc_tf.get(term, 0)
        if f > 0 and doc_length > 0:
            idf_score = self.idf(term)
            numerator = f * (self.k1 + 1)
            denominator = f + self.k1 * (1 - self.b + self.b * (doc_length / self.avgdl))
            score += idf_score * (numerator / denominator)
    
    return score


# attach to BM25 class
BM25.score = score

### Task 2c: Retrieve

Implement `retrieve()` to return the top-k most relevant documents for a given query, ranked by BM25 score.

In [11]:
def retrieve(self, query_text: str, k: int = 10) -> List[Tuple[str, float]]:
    """
    Return the top-k documents for a given query, ranked by BM25 score.
    Brute-force over all documents.
    
    Args:
        query_text: raw query string
        k:          number of documents to return

    Returns:
        List of (doc_id, score) tuples sorted by score descending

    Hints:
        - tokenize the query first using self.tokenize()
        - iterate over all documents in self.doc_tf.keys()
        - use self.score() to compute the BM25 score for each document
        - only keep documents with a non-zero score
    """
    query_tokens = self.tokenize(query_text)
    scores = []
    
    for doc_id in self.doc_tf.keys():
        score = self.score(query_tokens, doc_id)
        if score > 0:
            scores.append((doc_id, score))
    
    # Sort by score descending and return top-k
    scores.sort(key=lambda x: x[1], reverse=True)
    return scores[:k]

# attach to BM25 class
BM25.retrieve = retrieve

### Check Task 2: Implementing a BM25 retriever

Run the cell below to check your implementation. You should see:
- Total number of documents and average document length after building the index
- A ranked list of 10 documents for a sample query with their BM25 scores and titles


In [12]:
# Build index
bm25 = BM25()
print("Building index...")
bm25.build(dataset.docs_iter())
print(f"Done. N={bm25.N} docs, avgdl={bm25.avgdl:.2f}")

print('-'*120)

# Single query demo
q = next(dataset.queries_iter())
top10 = bm25.retrieve(q.text, k=10)
print("Query ID:", q.query_id, "Query Text:", q.text)
print('-'*50)
print("rank", "doc_id",  "score",  "title")
print('-'*50)
for rank, (doc_id, score) in enumerate(top10, 1):
    title = bm25.docs_store[doc_id]["title"]
    print(f"{rank:02d} {doc_id}  {score:.4f}  {title[:80]}")

Building index...
Done. N=1400 docs, avgdl=102.94
------------------------------------------------------------------------------------------------------------------------
Query ID: 1 Query Text: what similarity laws must be obeyed when constructing aeroelastic models
of heated high speed aircraft .
--------------------------------------------------
rank doc_id score title
--------------------------------------------------
01 51  21.9151  theory of aircraft structural models subjected to aerodynamic
heating and extern
02 486  21.2723  similarity laws for aerothermoelastic testing .
03 12  18.5659  some structural and aerelastic considerations of high
speed flight .
04 184  17.8901  scale models for thermo-aeroelastic research .
05 573  17.0003  viscous hypersonic similitude .
06 878  16.5020  experimental model techniques and equipment for flutter
investigations .
07 665  14.4273  on the theory of hypersonic gas flow with a power law
shock wave .
08 746  14.1964  aeroelastic problems in

## Task 3: BM25 Evaluator

### Task 3a: Implement evaluate function for "recall", "precision", "mrr", "ndcg"

In [ ]:
def evaluate(
    self,
    queries_iter: Iterable,
    qrels_iter: Iterable,
    k: int = 10,
    metrics: Optional[List[str]] = None,
) -> Dict[str, float]:
    """
    Compute evaluation metrics averaged over all queries.

    Args:
        queries_iter : iterable of queries (each has .query_id and .text)
        qrels_iter   : iterable of relevance judgments (each has .query_id, .doc_id, .relevance)
        k            : number of top documents to evaluate
        metrics      : list of metrics to compute, e.g. ["recall", "precision", "mrr", "ndcg"]
                       if None, compute all four

    Returns:
        Dict mapping metric name -> average score across all queries

    Note:
        Relevance:
            - treat relevance <= 0 as non-relevant (0)
            - treat relevance  > 0 as relevant (1)

        Metrics:
            Recall@k    = (number of relevant docs in top-k) / (total relevant docs for query)

            Precision@k = (number of relevant docs in top-k) / k

            MRR@k       = 1 / (rank of first relevant doc in top-k)
                        0 if no relevant doc found

            nDCG@k      = DCG@k / IDCG@k
                        DCG@k  = sum over rank i: rel_i / log2(i + 2)    (i is 0-indexed)
                        IDCG@k = DCG of the ideal ranking (sort all relevant docs by relevance desc)

    Hints:
        - define two helper functions inside evaluate():
            * normalize_relevance(rel) to convert relevance scores to 0 or 1 (binary)
            * dcg(rels) to compute DCG for a ranked relevance list
        - build a qrels dict: qrels[query_id][doc_id] = normalized relevance
        - skip queries that have no relevance judgments
        - use self.retrieve(query_text, k) to get ranked (doc_id, score) pairs
        - for duplicate qrel entries keep the maximum relevance
    """
        
    if metrics is None:
        metrics = ["recall", "precision", "mrr", "ndcg"]
    # Helper function to normalize relevance
    def normalize_relevance(rel):
        return 1 if rel > 0 else 0
    
    # Helper function to compute DCG
    def dcg(rels):
        return sum(rel / math.log2(i + 2) for i, rel in enumerate(rels))
    
    # Build qrels dict
    qrels = defaultdict(dict)
    for qrel in qrels_iter:
        qrels[qrel.query_id][qrel.doc_id] = max(qrels[qrel.query_id].get(qrel.doc_id, 0), normalize_relevance(qrel.relevance))
    
    # Evaluate each query
    metric_sums = defaultdict(float)
    query_count = 0
    for query in queries_iter:
        query_id = query.query_id
        query_text = query.text
        
        if query_id not in qrels:
            continue  # skip queries with no relevance judgments
        
        relevant_docs = {doc_id: rel for doc_id, rel in qrels[query_id].items() if rel > 0}
        num_relevant = len(relevant_docs)
        
        retrieved_docs = self.retrieve(query_text, k)
        retrieved_doc_ids = [doc_id for doc_id, _ in retrieved_docs]
        
        # Compute metrics
        num_retrieved_relevant = sum(1 for doc_id in retrieved_doc_ids if doc_id in relevant_docs)
        
        if "recall" in metrics:
            metric_sums["recall"] += num_retrieved_relevant / num_relevant if num_relevant > 0 else 0.0
        
        if "precision" in metrics:
            metric_sums["precision"] += num_retrieved_relevant / k
        
        if "mrr" in metrics:
            mrr_score = 0.0
            for rank, doc_id in enumerate(retrieved_doc_ids, 1):
                if doc_id in relevant_docs:
                    mrr_score = 1 / rank
                    break
            metric_sums["mrr"] += mrr_score
        
        if "ndcg" in metrics:
            rels = [relevant_docs.get(doc_id, 0) for doc_id in retrieved_doc_ids]
            dcg_score = dcg(rels)
            ideal_rels = sorted(relevant_docs.values(), reverse=True)[:k]
            idcg_score = dcg(ideal_rels)
            ndcg_score = dcg_score / idcg_score if idcg_score > 0 else 0.0
            metric_sums["ndcg"] += ndcg_score
        
        query_count += 1

    # Compute average metrics
    avg_metrics = {metric: metric_sums[metric] / query_count for metric in metrics}
    return avg_metrics


BM25.evaluate = evaluate

### Task 3b: Grid Search

Implement and perform grid search for k1 ∈ {0.60, 1.20, 1.80} and b ∈ {0.25, 0.50, 0.75}

In [14]:
def grid_search(
    self,
    queries_iter_factory,
    qrels_iter_factory,
    k: int = 10,
    k1_values: Tuple = (0.60, 1.20, 1.80),
    b_values: Tuple = (0.25, 0.50, 0.75),
) -> List[Dict]:
    """
    Evaluate all combinations of k1 and b, return results as a list of dicts.

    Args:
        queries_iter_factory : callable that returns a fresh queries iterator each time
        qrels_iter_factory   : callable that returns a fresh qrels iterator each time
        k                    : number of top documents to evaluate
        k1_values            : values of k1 to try
        b_values             : values of b to try

    Returns:
        List of dicts, one per (k1, b) combination:
        [
            {"k1": 0.60, "b": 0.25, "recall": ..., "precision": ..., "mrr": ..., "ndcg": ...},
            ...
        ]

    Note:
        - update self.k1 and self.b directly before each self.evaluate() call
        - pass factories/lambdas not iterators as ir_datasets iterators are single-pass,
          so call queries_iter_factory() and qrels_iter_factory() inside the loop to get a fresh iterator each time
    """
    results = []
    for k1 in k1_values:
        for b in b_values:
            self.k1 = k1
            self.b = b
            metrics = self.evaluate(queries_iter_factory(), qrels_iter_factory(), k)
            result = {"k1": k1, "b": b, **metrics}
            results.append(result)
    return results

# attach to BM25 class
BM25.grid_search = grid_search

#### Run the cell below to perform the grid search

In [15]:
# Grid search
# Make sure you have run the Check Task 2 cell first to initialize bm25
print("\nRunning grid search...")
results = bm25.grid_search(
    queries_iter_factory=dataset.queries_iter,
    qrels_iter_factory=dataset.qrels_iter,
    k=10,
)

# Display results table
import pandas as pd

df = pd.DataFrame(results)
df = df.set_index(["k1", "b"])
df.columns = ["Recall@10", "Precision@10", "MRR@10", "nDCG@10"] # change if your metric order is different
print("\n", df.to_string(float_format="%.4f"))





Running grid search...

           Recall@10  Precision@10  MRR@10  nDCG@10
k1  b                                             
0.6 0.25     0.3783        0.2182  0.5155   0.3628
    0.50     0.3888        0.2249  0.5251   0.3738
    0.75     0.3955        0.2302  0.5274   0.3806
1.2 0.25     0.3961        0.2302  0.5277   0.3806
    0.50     0.4076        0.2404  0.5367   0.3923
    0.75     0.4037        0.2400  0.5359   0.3924
1.8 0.25     0.4025        0.2373  0.5408   0.3889
    0.50     0.4108        0.2444  0.5476   0.3976
    0.75     0.4070        0.2453  0.5426   0.3982


#### TODO: Report and justify your choice of best configuration



**Which do you consider your best configuration:** 
The best configuration is k1 = 1.80 and b = 0.50.

**Why:** This configuration achieves the highest Recall@10 (0.4108) and MRR@10 (0.5476) across all tested combinations, while also delivering strong Precision@10 (0.2444) and nDCG@10 (0.3976). Although k1=1.8, b=0.75 slightly edges ahead on Precision (+0.0009) and nDCG (+0.0006), those differences are negligible, while k1=1.8, b=0.50 has a clear advantage in Recall (+0.0038) and MRR (+0.0050). Since Recall and MRR measure how completely and how early a system surfaces relevant documents, this configuration offers the best overall balance.

- **k1 = 1.80** (higher term-frequency saturation): Allows repeated term occurrences to contribute more to the score. This benefits the Cranfield collection, where technical documents use domain-specific terms repeatedly to signal topical relevance.
- **b = 0.50** (moderate length normalization): Provides a balance between penalizing overly long documents and not penalizing enough. The Cranfield corpus has moderate variation in document length, so an intermediate normalization avoids over- or under-correcting for length.

## Task 4: Error Analysis

### TODO:
### Pick two queries, one where BM25 retrieved great documents, one where BM25 failed the retrieval.


#### Query 1 — Successful Retrieval

**Query ID:** 25

**Query Text:** "does a practical flow follow the theoretical concepts for the interaction between adjacent blade rows of a supersonic cascade."

**Top 5 Retrieved Documents:**

| Rank | Doc ID | Score  | Title | Relevant? |
|------|--------|--------|-------|-----------|
| 1 | 277 | 25.970 | Study of flow conditions and deflection angle at exit of two-dimensional cascade of turbine blades | Yes |
| 2 | 214 | 21.278 | On the testing of supersonic compressor cascades | Yes |
| 3 | 216 | 19.466 | The supersonic axial flow compressor | Yes |
| 4 | 215 | 19.412 | The test performance of highly loaded turbine stages designed for high pressure ratio | Yes |
| 5 | 213 | 19.302 | The performance of supersonic turbine nozzles | Yes |

**Does the ranking make sense?** Yes, all 5 retrieved documents are relevant (5/5 precision, 5 out of 9 total relevant docs). The query uses specific technical terms ("supersonic", "cascade", "blade rows", "flow") that directly match the vocabulary in these documents. BM25 excels here because the query and documents share a tight, domain-specific lexical overlap. The top-ranked document (277) discusses flow in cascades of turbine blades, which is the most direct match.

**Proposed Improvement:** While BM25 performs well, it retrieved only 5 of 9 relevant documents in the top 5. The remaining relevant documents may use synonyms or related terms (e.g., "compressor" vs. "cascade"). Query expansion using pseudo-relevance feedback (e.g., adding terms from top-ranked documents back into the query) could help surface the remaining 4 relevant documents.

#### Query 2 — Unsuccessful Retrieval

**Query ID:** 219

**Query Text:** "what are the general effects on flow fields when the reynolds number is small."

**Top 5 Retrieved Documents:**

| Rank | Doc ID | Score  | Title | Relevant? |
|------|--------|--------|-------|-----------|
| 1 | 1221 | 13.399 | Steady flow of conducting fluids in channels under transverse magnetic fields... | No |
| 2 | 208  | 12.523 | The hall effect in the viscous flow of ionized gas between parallel plates... | No |
| 3 | 1222 | 12.239 | Axisymmetric magnetohydrodynamic channel flow | No |
| 4 | 992  | 12.233 | The effects of a small jet of air exhausting from the nose of a body of revolution in supersonic flow | No |
| 5 | 993  | 11.625 | The extent of the jet interference flow fields | No |

**Does the ranking make sense?** No, 0 out of 5 retrieved documents are relevant (0/18 total relevant docs found). The query is broad and conceptual ("general effects", "flow fields", "reynolds number is small"), using common terms that appear in many unrelated fluid dynamics documents. BM25 matches on generic tokens like "flow", "field", "effect", and "small", which occur frequently across the corpus, leading to poor discrimination. Meanwhile, the actually relevant documents (e.g., doc 666: "blunt body heat transfer at hypersonic speed and low reynolds numbers", doc 667: "hypersonic shock layer theory of the stagnation region at low reynolds number") use specific terms like "hypersonic", "heat transfer", "stagnation", and "blunt body" that do not appear in the query at all.

**Proposed Improvement:** This is a vocabulary mismatch problem. BM25 relies purely on lexical overlap and cannot bridge the gap between the query's abstract phrasing and the documents' specific terminology. Possible improvements:
1. **Query expansion:** Use pseudo-relevance feedback or a thesaurus to add domain-specific terms (e.g., "hypersonic", "stagnation", "blunt body") to the query.
2. **Semantic retrieval:** Use dense retrieval (e.g., embedding-based models) that can capture semantic similarity beyond exact term matching, linking "low reynolds number effects" to documents about specific low-Reynolds-number phenomena.
3. **Hybrid retrieval:** Combine BM25 with a semantic model to leverage both lexical precision and semantic understanding.

## Appendix: Error Analysis Code

The code below was used to identify the best and worst performing queries for the error analysis in Task 4.


In [ ]:
# Per-query evaluation to find best/worst queries
bm25.k1 = 1.80
bm25.b = 0.50

# Build qrels dict
qrels_dict = defaultdict(dict)
for qrel in dataset.qrels_iter():
    rel = 1 if qrel.relevance > 0 else 0
    qrels_dict[qrel.query_id][qrel.doc_id] = max(
        qrels_dict[qrel.query_id].get(qrel.doc_id, 0), rel
    )

per_query = []
for query in dataset.queries_iter():
    qid = query.query_id
    if qid not in qrels_dict:
        continue
    relevant = {d for d, r in qrels_dict[qid].items() if r > 0}
    top5 = bm25.retrieve(query.text, k=5)
    retrieved_ids = [d for d, _ in top5]
    hits = sum(1 for d in retrieved_ids if d in relevant)
    mrr = 0.0
    for rank, d in enumerate(retrieved_ids, 1):
        if d in relevant:
            mrr = 1.0 / rank
            break
    precision = hits / 5
    recall = hits / len(relevant) if relevant else 0
    per_query.append({
        "qid": qid, "text": query.text, "precision": precision,
        "recall": recall, "mrr": mrr, "hits": hits, "total_rel": len(relevant),
        "top5": top5
    })

# Sort by combined score (precision + recall + mrr)
per_query.sort(key=lambda x: x["precision"] + x["recall"] + x["mrr"], reverse=True)

print("=== TOP 5 BEST QUERIES ===")
for q in per_query[:5]:
    print(f"QID={q['qid']}  P@5={q['precision']:.2f}  R@5={q['recall']:.2f}  MRR={q['mrr']:.2f}  "
          f"hits={q['hits']}/{q['total_rel']}  {q['text'][:80]}")

print("\n=== TOP 5 WORST QUERIES ===")
for q in per_query[-5:]:
    print(f"QID={q['qid']}  P@5={q['precision']:.2f}  R@5={q['recall']:.2f}  MRR={q['mrr']:.2f}  "
          f"hits={q['hits']}/{q['total_rel']}  {q['text'][:80]}")


In [ ]:
# Detail for successful query (QID=25)
q25_text = next(q for q in dataset.queries_iter() if q.query_id == "25").text
print(f"Query 25: {q25_text}")
top5_25 = bm25.retrieve(q25_text, k=5)
rel25 = {d for d, r in qrels_dict["25"].items() if r > 0}
for rank, (doc_id, sc) in enumerate(top5_25, 1):
    t = bm25.docs_store[doc_id]["title"][:90]
    tag = "REL" if doc_id in rel25 else "NOT"
    print(f"  {rank}. [{tag}] {doc_id} ({sc:.3f}) {t}")
print(f"Hits: {sum(1 for d,_ in top5_25 if d in rel25)}/5, Total relevant: {len(rel25)}")

print()

# Detail for failed query (QID=219)
q219_text = next(q for q in dataset.queries_iter() if q.query_id == "219").text
print(f"Query 219: {q219_text}")
top5_219 = bm25.retrieve(q219_text, k=5)
rel219 = {d for d, r in qrels_dict["219"].items() if r > 0}
for rank, (doc_id, sc) in enumerate(top5_219, 1):
    t = bm25.docs_store[doc_id]["title"][:90]
    tag = "REL" if doc_id in rel219 else "NOT"
    print(f"  {rank}. [{tag}] {doc_id} ({sc:.3f}) {t}")
print(f"Hits: {sum(1 for d,_ in top5_219 if d in rel219)}/5, Total relevant: {len(rel219)}")
print("Some relevant docs not retrieved:")
for did in list(rel219)[:3]:
    print(f"  {did}: {bm25.docs_store[did]['title'][:90]}")
